In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os, re, json, warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve, roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss,
)
import joblib

warnings.filterwarnings('ignore', category = UserWarning)

@dataclass
class Config:
  data_dir: str = "/content/drive/MyDrive/submission_data"
  train_file: str = "/content/drive/MyDrive/submission_data/train_data.csv"
  val_file: str = "/content/drive/MyDrive/submission_data/val_data.csv"
  test_file: str = "/content/drive/MyDrive/submission_data/test_data.csv"

  out_dir: str = "/content/drive/MyDrive/submission/lexicon_final"
  feature_cols: Tuple[str, ...] = ("Tone", "affect", "posemo", "negemo", "anx", "anger", "sad")
  label_cols: Tuple[str, ...] = ("labelanx", "labelanger", "labelsad")
  calib_method: str = "sigmoid"
  random_state: int = 1234
  domain_col: str = "domain"
  extra_drop: Tuple[str, ...] = tuple()

thresh_policy = {
    "label_anx": {"mode": "best_f1",          "p_target":None},
    "label_anger": {"mode": "constrained_f1", "p_target":0.7},
    "label_sad": {"mode": "best_f1",          "p_target": None},
}

sweep_targets = (0.6, 0.7, 0.8, 0.9)

def _to_numeric_series(s: pd.Series) -> pd.Series:
  if s.dtype == object:
    s = s.astype(str).str.replace(",",".", regex = False)
  return pd.to_numeric(s, errors = "coerce")

def load_df(path: str, feature_cols: list[str], label_cols: list[str]) -> pd.DataFrame:
  df = pd.read_csv(path)

  missing = [col for col in feature_cols + label_cols if col not in df.columns]
  if missing:
    raise ValueError(
        f"Missing columns in {path}: {missing}\n"
        f"Columns found (sample): {list(df.columns)[:50]}"
        )
  # features to numeric
  for col in feature_cols:
    df[col] = _to_numeric_series(df[col])

  # labels to int 0/1
  for y in label_cols:
    df[y] = pd.to_numeric(df[y], errors = "coerce").fillna(0).astype(int)

  return df

def base_pipeline(cfg: Config) -> Pipeline:
  return Pipeline(steps=[
      ("imputer", SimpleImputer(strategy = "median")),
      ("scaler", StandardScaler()),
      ("classifier", LogisticRegression(
          solver = "liblinear",
          class_weight = "balanced",
          penalty = "l2",
          random_state = cfg.random_state,
          max_iter = 2000,
      ))
  ])

def features_for_label(cfg: Config, label: str) -> List[str]:
   drop_map = {"labelanx": "anx","labelanger":"anger", "labelsad":"sad"}
   drop = set(cfg.extra_drop)
   if label in drop_map:
     drop.add(drop_map[label])
   return [c for c in cfg.feature_cols if c not in drop]

def choose_threshold_best_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
  precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred)
  p2, r2, thr2 = precisions[1:], recalls[1:], thresholds
  if len(thr2) == 0:
    return 0.5
  f1 = 2 * p2 * r2 / (p2 + r2 + 1e-12)
  return float(thr2[int(np.nanargmax(f1))])

def choose_threshold_constrained_f1(y_true: np.ndarray, y_pred: np.ndarray, p_target: float) -> float:
  prec, rec, thr = precision_recall_curve(y_true, y_pred)
  prec2, rec2, thr2 = prec[1:], rec[1:], thr
  if len(thr2) == 0:
    return 0.5
  f1 = 2 * prec2 * rec2 / (prec2 + rec2 + 1e-12)
  ok = np.where(prec2 >= p_target)[0]
  if len(ok) ==0:
    return choose_threshold_best_f1(y_true, y_pred)
  best = ok[int(np.nanargmax(f1[ok]))]
  return float(thr2[best])

def confusion_parts(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
  tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
  return {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}

def metrics(y_true: np.ndarray, p: np.ndarray, thr: float) -> Dict[str, float]:
  y_pred = (p >= thr).astype(int)
  return {
      "pr_auc": float(average_precision_score(y_true, p)) if len(np.unique(y_true)) > 1 else float("nan"),
      "roc_auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) > 1 else float("nan"),
      "f1": float(f1_score(y_true, y_pred, zero_division=0)),
      "precision": float(precision_score(y_true, y_pred, zero_division=0)),
      "recall": float(recall_score(y_true, y_pred, zero_division=0)),
      "brier": float(brier_score_loss(y_true, p)),
      "threshold": float(thr),
      "positives_pct": float(100.0 * y_true.mean()),
      "n": int(len(y_true)),
      **confusion_parts(y_true, y_pred),
  }

def prob_summary(y: np.ndarray, p: np.ndarray) -> Dict[str, Dict[str, float]]:
  pos = p[y ==1]
  neg = p[y ==0]

  def quantiles(arr):
    if len(arr) == 0:
      return {"n":0, "p50": np.nan, "p90": np.nan, "p99": np.nan}
    return {
        "n": len(arr),
        "p50": np.percentile(arr, 50),
        "p90": np.percentile(arr, 90),
        "p99": np.percentile(arr, 99),
    }
  return {
      "positive": quantiles(pos),
      "negative": quantiles(neg),
  }

def domain_metrics(df: pd.DataFrame, y_col: str, p_col: str, thr: float, domain_col: str) -> Dict[str, Dict[str, float]]:
  out = {}
  for domain, df_domain in df.groupby(domain_col):
    y = df_domain[y_col].to_numpy().astype(int)
    p = df_domain[p_col].to_numpy().astype(float)
    out[str(domain)] = metrics(y, p, thr)
  return out

In [8]:
def run(cfg: Config):
  os.makedirs(cfg.out_dir, exist_ok=True)

  train_df = load_df(cfg.train_file, list(cfg.feature_cols), list(cfg.label_cols))
  val_df = load_df(cfg.val_file, list(cfg.feature_cols), list(cfg.label_cols))
  test_df = load_df(cfg.test_file, list(cfg.feature_cols), list(cfg.label_cols))

  has_domain = (
      cfg.domain_col in train_df.columns and
      cfg.domain_col in val_df.columns and
      cfg.domain_col in test_df.columns
  )
  if not has_domain:
    print(f"No '{cfg.domain_col}' column found in train/val/test data")

  all_results ={}

  for label in cfg.label_cols:
    features = features_for_label(cfg, label)
    policy = thresh_policy.get(label, {"mode": "best_f1", "p_target": None})

    x_train = train_df[features].to_numpy(float)
    x_val = val_df[features].to_numpy(float)
    x_test = test_df[features].to_numpy(float)

    y_train = train_df[label].to_numpy().astype(int)
    y_val = val_df[label].to_numpy().astype(int)
    y_test = test_df[label].to_numpy().astype(int)

    base = base_pipeline(cfg)
    base.fit(x_train, y_train)

    calibrated = CalibratedClassifierCV(
        estimator = base,
        cv = "prefit",
        method = cfg.calib_method,
    )
    calibrated.fit(x_val, y_val)

    p_val = calibrated.predict_proba(x_val)[:,1]
    p_test = calibrated.predict_proba(x_test)[:,1]

    thr_best = choose_threshold_best_f1(y_val, p_val)

    #policy threshold
    mode = policy.get("mode", "best_f1")
    p_target = policy.get("p_target", None)

    if mode == "constrained_f1" and p_target is not None:
      thr_policy = choose_threshold_constrained_f1(y_val, p_val, float(p_target))
    else:
      # fallback (covers best_f1 and constrained_f1 with p_target=None)
      thr_policy = thr_best

    # sweep
    if mode == "constrained_f1" and p_target is not None:
      sweep_thr = {str(pt): choose_threshold_constrained_f1(y_val, p_val, float(pt))for pt in sweep_targets}
      sweep_block = {
          str(pt):{
              "thr": float(sweep_thr[str(pt)]),
              "val": metrics(y_val, p_val, sweep_thr[str(pt)]),
              "test": metrics(y_test, p_test, sweep_thr[str(pt)]),
          } for pt in sweep_targets
      }
    else:
      sweep_thr = {}
      sweep_block = {}

    pred_val_df = val_df.copy()
    pred_val_df["p"] = p_val
    pred_test_df = test_df.copy()
    pred_test_df["p"] = p_test

    print(f"\n {'='*50}")
    print(f"Label: {label}")
    print(f"Features: {features}")
    print(f"Calibration method: {cfg.calib_method}")
    print(f"Policy: {policy} | thr_policy: {thr_policy:.5f} | thr_best: {thr_best:.5f}")

    results ={

        "label": label,
        "features_used": features,
        "calib_method": cfg.calib_method,
        "threshold_policy": policy,
        "val_prob_summary": prob_summary(y_val, p_val),

        "best_f1": {
            "thr": float(thr_best),
            "val": metrics(y_val, p_val,  thr_best),
            "test": metrics(y_test, p_test, thr_best),
        },

        "policy": {
            "thr": float(thr_policy),
            "val": metrics(y_val, p_val, thr_policy),
            "test": metrics(y_test, p_test, thr_policy),
        },

        "sweep": sweep_block,
    }

    results["resolved_threshold"] = {
        "mode":mode if (mode=="constrained_f1" and p_target is not None)else "best_f1",
        "p_target": p_target if (mode=="constrained_f1" and p_target is not None) else None
        }

    pred_val_df[f"{label}_pred_policy"] = (p_val >= thr_policy).astype(int)
    pred_test_df[f"{label}_pred_policy"] = (p_test >= thr_policy).astype(int)

    if has_domain:
      results["test_domain_policy"] = domain_metrics(
          pred_test_df, label, "p", thr_policy, cfg.domain_col
          )

    all_results[label] = results

    print("Val (policy):", results["policy"]["val"])
    print("Test (policy):", results["policy"]["test"])
    print("Val (bestF1):", results["best_f1"]["val"])
    print("Test (bestF1):", results["best_f1"]["test"])
    print("Val sweep:", {k: (v["val"]["precision"],  v["val"]["recall"], v["val"]["f1"]) for k, v in results["sweep"].items()})
    print("Test sweep:", {k: (v["test"]["precision"], v["test"]["recall"], v["test"]["f1"]) for k,v in results["sweep"].items()})

    if has_domain:
        print("Test by domain (policy):", {
            k: {"pr_auc": v["pr_auc"], "f1": v["f1"], "precision": v["precision"], "recall": v["recall"]}
            for k, v in results["test_domain_policy"].items()
        })

    label_dir = os.path.join(cfg.out_dir, label)
    os.makedirs(label_dir, exist_ok=True)
    joblib.dump(base, os.path.join(label_dir, "base_model.joblib"))
    joblib.dump(calibrated, os.path.join(label_dir, "calibrated_model.joblib"))
    with open(os.path.join(label_dir, "results.json"), "w") as f:
      json.dump(results, f, indent = 2)

    pred_val_df.to_csv(os.path.join(label_dir, "val_predictions.csv"), index = False)
    pred_test_df.to_csv(os.path.join(label_dir, "test_predictions.csv"), index = False)

  with open(os.path.join(cfg.out_dir, "all_results.json"), "w") as f:
    json.dump(all_results, f, indent = 2)

  print("\nSaved to:", cfg.out_dir)

def main():
    cfg = Config()
    run(cfg)

if __name__ == "__main__":
    main()



Label: labelanx
Features: ['Tone', 'affect', 'posemo', 'negemo', 'anger', 'sad']
Calibration method: sigmoid
Policy: {'mode': 'best_f1', 'p_target': None} | thr_policy: 0.08357 | thr_best: 0.08357
Val (policy): {'pr_auc': 0.3683224721389058, 'roc_auc': 0.9082986545837556, 'f1': 0.46173308032890575, 'precision': 0.32272325375773653, 'recall': 0.8111111111111111, 'brier': 0.057934432426736876, 'threshold': 0.08357276626972852, 'positives_pct': 7.4490978314848535, 'n': 6041, 'tp': 365, 'tn': 4825, 'fp': 766, 'fn': 85}
Test (policy): {'pr_auc': 0.6852577698401485, 'roc_auc': 0.9322736814496557, 'f1': 0.7454409193105171, 'precision': 0.604783137413863, 'recall': 0.9713541666666666, 'brier': 0.09842987363478412, 'threshold': 0.08357276626972852, 'positives_pct': 20.474540122633965, 'n': 30008, 'tp': 5968, 'tn': 19964, 'fp': 3900, 'fn': 176}
Val (bestF1): {'pr_auc': 0.3683224721389058, 'roc_auc': 0.9082986545837556, 'f1': 0.46173308032890575, 'precision': 0.32272325375773653, 'recall': 0.811